# CI/CD Pipeline: Diabetes Readmission Prediction

**AAI-540 Group 5**

This notebook builds an automated SageMaker Pipeline (DAG) that preprocesses data, trains an XGBoost model, evaluates it, and registers it to the Model Registry only if it meets a quality threshold. Every pipeline run is versioned and repeatable, which is the basis of our CI/CD process.

**Pipeline flow:** Preprocess → Train → Evaluate → Condition Check → Register Model


In [1]:
%pip install -q "sagemaker>=2.200,<3"

Note: you may need to restart the kernel to use updated packages.


## 1. Environment Setup

We pin the SageMaker SDK below v3 for compatibility, then create a `PipelineSession`. Steps built with a pipeline session are defined but not run right away. They run only when the pipeline itself executes.


In [ ]:
# Standard libraries
import os
import json
import time
from time import sleep

# Data
import pandas as pd

# AWS / SageMaker core
import boto3
import sagemaker
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics

# Processing
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor

# Pipeline building blocks
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import (
    ParameterString,
    ParameterInteger,
    ParameterFloat,
)
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.fail_step import FailStep

# Session setup
region = boto3.Session().region_name
role = sagemaker.get_execution_role()
pipeline_session = PipelineSession()
bucket = pipeline_session.default_bucket()
prefix = "diabetes-readmission/pipeline"
s3 = boto3.client("s3")
sm = boto3.client("sagemaker")

print(region, role, bucket, sep="\n")
print("SageMaker SDK:", sagemaker.__version__)

print("Working dir:", os.getcwd())

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


us-east-1
arn:aws:iam::769592409248:role/LabRole
sagemaker-us-east-1-769592409248
SageMaker SDK: 2.257.7
Working dir: /home/sagemaker-user/diabetes-readmission-prediction/notebooks/setup


In [ ]:
for fg in sm.list_feature_groups(MaxResults=20)["FeatureGroupSummaries"]:
    print(
        fg["FeatureGroupName"],
        "|",
        fg["FeatureGroupStatus"],
        "|",
        fg.get("OfflineStoreStatus", {}).get("Status"),
    )
    desc = sm.describe_feature_group(FeatureGroupName=fg["FeatureGroupName"])
    print(
        "   Offline S3:",
        desc.get("OfflineStoreConfig", {})
        .get("S3StorageConfig", {})
        .get("ResolvedOutputS3Uri"),
    )
    print(
        "   Glue table:",
        desc.get("OfflineStoreConfig", {})
        .get("DataCatalogConfig", {})
        .get("TableName"),
    )
    print("   # features:", len(desc["FeatureDefinitions"]))

diabetes-feature-group-25-06-12-59 | Created | Active
   Offline S3: s3://sagemaker-us-east-1-769592409248/carepath-analytics/feature-store/769592409248/sagemaker/us-east-1/offline-store/diabetes-feature-group-25-06-12-59-1790316779/data
   Glue table: diabetes_feature_group_25_06_12_59_1790316779
   # features: 25
diabetes-feature-group-25-06-00-04 | Created | Active
   Offline S3: s3://sagemaker-us-east-1-769592409248/carepath-analytics/feature-store/769592409248/sagemaker/us-east-1/offline-store/diabetes-feature-group-25-06-00-04-1790316004/data
   Glue table: diabetes_feature_group_25_06_00_04_1790316004
   # features: 25
diabetes-feature-group-25-05-53-01 | Created | Active
   Offline S3: s3://sagemaker-us-east-1-769592409248/carepath-analytics/feature-store/769592409248/sagemaker/us-east-1/offline-store/diabetes-feature-group-25-05-53-01-1790315582/data
   Glue table: diabetes_feature_group_25_05_53_01_1790315582
   # features: 25
diabetes-feature-group-25-05-44-51 | Created | Ac

In [ ]:
feature_store_table_name = desc["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"]

## 2. Locate Input Data

We find the cleaned diabetes dataset in S3 that our data ingestion step produced. This becomes the input to the pipeline's Preprocess step.


In [13]:
s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=bucket):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if "diabet" in key.lower() and key.endswith((".csv", ".parquet")):
            print(f"s3://{bucket}/{key}  ({obj['Size']/1e6:.1f} MB)")

s3://sagemaker-us-east-1-769592409248/carepath-analytics/feature-store/769592409248/sagemaker/us-east-1/offline-store/diabetes-feature-group-16-23-11-09-1792192269/data/year=2026/month=10/day=16/hour=23/20261016T231108Z_083cb19c472d7592.parquet  (0.0 MB)
s3://sagemaker-us-east-1-769592409248/carepath-analytics/feature-store/769592409248/sagemaker/us-east-1/offline-store/diabetes-feature-group-16-23-11-09-1792192269/data/year=2026/month=10/day=16/hour=23/20261016T231108Z_0d3a13d903b0bdb0.parquet  (0.0 MB)
s3://sagemaker-us-east-1-769592409248/carepath-analytics/feature-store/769592409248/sagemaker/us-east-1/offline-store/diabetes-feature-group-16-23-11-09-1792192269/data/year=2026/month=10/day=16/hour=23/20261016T231108Z_0ec75820aa3afb19.parquet  (0.0 MB)
s3://sagemaker-us-east-1-769592409248/carepath-analytics/feature-store/769592409248/sagemaker/us-east-1/offline-store/diabetes-feature-group-16-23-11-09-1792192269/data/year=2026/month=10/day=16/hour=23/20261016T231108Z_0fce2ab90531442

## 3. Inspect Processed Data

We check the shape, columns, and target distribution of the processed dataset so the pipeline's preprocessing script handles it correctly.


In [14]:
input_data_uri = f"s3://{bucket}/carepath-analytics/processed/diabetes_processed.csv"

df = pd.read_csv(input_data_uri)
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:\n", df.dtypes.value_counts())
print("\nTarget candidates:")
for col in df.columns:
    if "readmit" in col.lower() or "target" in col.lower() or "label" in col.lower():
        print(col, df[col].value_counts().to_dict())

Shape: (101766, 57)

Columns: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted', 'readmitted_30d', 'admission_type', 'discharge_disposition', 'admission_source', 'diag_1_group', 'diag_2_group', 'diag_3_group']

Dtypes:
 object    43


## 3b. Feature Store Extraction Script

The pipeline trains from the **SageMaker Feature Store**, not from a static file. This script runs in the first pipeline step. It uses Athena to query the Feature Store's offline table, keeps only the latest record for each encounter (removing duplicates and deleted records), and saves the result as a CSV for preprocessing.


In [15]:
%%writefile extract_features.py
import argparse
import os
import time

import boto3

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--database", required=True)
    parser.add_argument("--table", required=True)
    parser.add_argument("--staging", required=True)
    parser.add_argument("--region", required=True)
    args = parser.parse_args()

    athena = boto3.client("athena", region_name=args.region)
    s3 = boto3.client("s3", region_name=args.region)

    query = f"""
        SELECT * FROM (
            SELECT *, row_number() OVER (
                PARTITION BY encounter_id
                ORDER BY event_time DESC, api_invocation_time DESC
            ) AS rn
            FROM "{args.table}"
            WHERE NOT is_deleted
        ) WHERE rn = 1
    """

    qid = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={"Database": args.database},
        ResultConfiguration={"OutputLocation": args.staging},
    )["QueryExecutionId"]

    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        if status["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(3)

    if status["State"] != "SUCCEEDED":
        raise RuntimeError(f"Athena query {status['State']}: {status.get('StateChangeReason')}")

    result_uri = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["ResultConfiguration"]["OutputLocation"]
    bucket, key = result_uri.replace("s3://", "").split("/", 1)

    out_dir = "/opt/ml/processing/output"
    os.makedirs(out_dir, exist_ok=True)
    s3.download_file(bucket, key, f"{out_dir}/features.csv")
    print("Extracted Feature Store data to features.csv from", result_uri)

Overwriting extract_features.py


## 4. Preprocessing Script

This script runs in the pipeline's Processing step and reproduces the team's training notebook (`02-initialize-model.ipynb`):

- Uses the same **19 model features** (11 categorical, 8 numeric). Admission, discharge, and source are stored as IDs in the Feature Store, so they are one-hot encoded as categories.
- **Patient-level, stratified 70/15/15 split** with `random_state=42`, so no patient appears in more than one split
- **ColumnTransformer:** one-hot encoding for categorical features and standard scaling for numeric features, fit on training data only
- Saves train, validation, and test CSVs (label first, no header) for built-in XGBoost
- Saves the test set's **race and gender** separately for the RecallGap fairness gate
- Saves the fitted **preprocessor** so the deployment step can transform new patient data the same way
- Restores original column names, since Athena returns them in lowercase (for example `a1cresult` → `A1Cresult`)

> Note: `%%writefile` scripts run in their own container, so each script has its own imports.


In [16]:
%%writefile preprocess.py
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE = "/opt/ml/processing"
RANDOM_STATE = 42
TARGET = "readmitted_30d"

CATEGORICAL = [
    "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "diag_1_group",
    "diag_2_group",
    "diag_3_group",
    "max_glu_serum",
    "A1Cresult",
    "change",
    "diabetesMed",
]
NUMERIC = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]
SENSITIVE = ["race", "gender", "age"]


def main():
    df = pd.read_csv(f"{BASE}/input/features.csv", keep_default_na=False)

    # Athena lowercases column names; restore the original names
    expected = (
        CATEGORICAL + NUMERIC + SENSITIVE + [TARGET, "patient_nbr", "encounter_id"]
    )
    df = df.rename(columns={c.lower(): c for c in expected})
    print("Feature Store rows:", df.shape)

    for c in CATEGORICAL + SENSITIVE:
        df[c] = df[c].astype(str).replace({"": "Unknown", "?": "Unknown"})
    for c in NUMERIC + [TARGET]:
        df[c] = pd.to_numeric(df[c])
    df[TARGET] = df[TARGET].astype(int)

    # Patient-level stratified split (same as train-model.ipynb)
    patient_ids = df["patient_nbr"].astype(str)
    patient_table = (
        pd.DataFrame({"patient_nbr": patient_ids, "target": df[TARGET]})
        .groupby("patient_nbr", as_index=False)["target"]
        .max()
    )
    train_p, temp_p = train_test_split(
        patient_table,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=patient_table["target"],
    )
    val_p, test_p = train_test_split(
        temp_p, test_size=0.50, random_state=RANDOM_STATE, stratify=temp_p["target"]
    )
    masks = {
        "train": patient_ids.isin(set(train_p["patient_nbr"])),
        "validation": patient_ids.isin(set(val_p["patient_nbr"])),
        "test": patient_ids.isin(set(test_p["patient_nbr"])),
    }

    # Fit preprocessor on train only (same as train-model.ipynb)
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", sparse_output=True),
                CATEGORICAL,
            ),
            ("numeric", StandardScaler(), NUMERIC),
        ],
        remainder="drop",
    )
    preprocessor.fit(df.loc[masks["train"], CATEGORICAL + NUMERIC])
    print("Transformed feature count:", len(preprocessor.get_feature_names_out()))

    for name, mask in masks.items():
        part = df.loc[mask]
        X_t = preprocessor.transform(part[CATEGORICAL + NUMERIC]).toarray()
        out = np.column_stack([part[TARGET].to_numpy(), X_t])
        os.makedirs(f"{BASE}/{name}", exist_ok=True)
        np.savetxt(f"{BASE}/{name}/{name}.csv", out, delimiter=",", fmt="%.6g")
        print(f"{name}: {part.shape[0]} rows, positive rate {part[TARGET].mean():.4f}")

    # Sensitive attributes for the RecallGap gate (same row order as test.csv)
    os.makedirs(f"{BASE}/sensitive", exist_ok=True)
    df.loc[masks["test"], SENSITIVE].to_csv(
        f"{BASE}/sensitive/test_sensitive.csv", index=False
    )

    # Save fitted preprocessor for deployment
    os.makedirs(f"{BASE}/preprocessor", exist_ok=True)
    joblib.dump(preprocessor, f"{BASE}/preprocessor/preprocessor.joblib")


if __name__ == "__main__":
    main()


Overwriting preprocess.py


In [17]:
!python -m py_compile preprocess.py && echo "Syntax OK"

Syntax OK


## 5. Pipeline Parameters, Feature Store Extraction, and Preprocessing Steps

**Parameters** make the pipeline reusable. A CI/CD run can point to a different Feature Store table or change any quality gate threshold without code changes.

**Quality gate thresholds** match the team's automated evaluation (`automated-model-evaluation.ipynb`), plus a new **RecallGap** fairness gate:
| Gate | Default |
|---|---|
| Recall | ≥ 0.70 |
| PR-AUC | ≥ 0.20 |
| ROC-AUC | ≥ 0.65 |
| Precision | ≥ 0.15 |
| Max recall gap (race, gender) | ≤ 0.10 |
| Beat majority-class benchmark (PR-AUC, recall) | required |

**Step 1, ExtractFromFeatureStore:** queries the Feature Store offline table with Athena. Caching is off for this step, so every run pulls the latest data.

**Step 2, PreprocessDiabetesData:** runs `preprocess.py` on the extracted data and writes train, validation, and test splits, test-set sensitive attributes, and the fitted preprocessor.


In [ ]:
# Feature Store source
feature_store_db = "sagemaker_featurestore"
feature_group_table = ParameterString(
    name="FeatureGroupTable", default_value=feature_store_table_name
)
athena_staging = f"s3://{bucket}/athena/staging"

# Instance types
processing_instance_type = ParameterString(
    name="ProcessingInstanceType", default_value="ml.m5.large"
)
training_instance_type = ParameterString(
    name="TrainingInstanceType", default_value="ml.m5.large"
)

# Quality gate thresholds
min_recall = ParameterFloat(name="MinRecall", default_value=0.70)
min_pr_auc = ParameterFloat(name="MinPRAUC", default_value=0.20)
min_roc_auc = ParameterFloat(name="MinROCAUC", default_value=0.65)
min_precision = ParameterFloat(name="MinPrecision", default_value=0.15)
max_recall_gap = ParameterFloat(name="MaxRecallGap", default_value=0.10)

model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)

cache_config = CacheConfig(enable_caching=True, expire_after="PT12H")

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="diabetes-pipeline",
    sagemaker_session=pipeline_session,
)

# Step 1: Extract from Feature Store
extract_args = sklearn_processor.run(
    code="extract_features.py",
    arguments=[
        "--database",
        feature_store_db,
        "--table",
        feature_group_table,
        "--staging",
        athena_staging,
        "--region",
        region,
    ],
    outputs=[
        ProcessingOutput(output_name="features", source="/opt/ml/processing/output")
    ],
)
step_extract = ProcessingStep(name="ExtractFromFeatureStore", step_args=extract_args)

# Step 2: Preprocess
processor_args = sklearn_processor.run(
    code="preprocess.py",
    inputs=[
        ProcessingInput(
            source=step_extract.properties.ProcessingOutputConfig.Outputs[
                "features"
            ].S3Output.S3Uri,
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(
            output_name="validation", source="/opt/ml/processing/validation"
        ),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test"),
        ProcessingOutput(
            output_name="sensitive", source="/opt/ml/processing/sensitive"
        ),
        ProcessingOutput(
            output_name="preprocessor", source="/opt/ml/processing/preprocessor"
        ),
    ],
)
step_process = ProcessingStep(
    name="PreprocessDiabetesData", step_args=processor_args, cache_config=cache_config
)

print("Steps defined:", step_extract.name, "→", step_process.name)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: SKLearnProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Steps defined: ExtractFromFeatureStore → PreprocessDiabetesData


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:340: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## 6. Training Step

Trains SageMaker's built-in XGBoost (container 3.0-5, the same version used in the Model Registry and deployment) with the team's final hyperparameters from `02-initialize-model.ipynb`:

- 500 rounds, max_depth 6, learning rate 0.03, subsample 0.9, colsample_bytree 0.9
- `eval_metric = aucpr`, since PR-AUC is the team's primary metric for this imbalanced problem
- `scale_pos_weight = 8`, about the ratio of negatives to positives in the training data
- `tree_method = hist`, as in the team notebook

The step reads the Preprocess step's outputs directly, which creates the next edge in the DAG.


In [ ]:
model_output_path = f"s3://{bucket}/{prefix}/model"

xgb_image = image_uris.retrieve(framework="xgboost", region=region, version="3.0-5")

xgb_estimator = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_type=training_instance_type,
    instance_count=1,
    output_path=model_output_path,
    base_job_name="diabetes-xgb-train",
    sagemaker_session=pipeline_session,
)

xgb_estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="aucpr",
    num_round=500,
    max_depth=6,
    eta=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=8,
    tree_method="hist",
    seed=42,
)

train_args = xgb_estimator.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
                "train"
            ].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs[
                "validation"
            ].S3Output.S3Uri,
            content_type="text/csv",
        ),
    }
)

step_train = TrainingStep(
    name="TrainXGBoostModel", step_args=train_args, cache_config=cache_config
)

print("XGBoost image:", xgb_image)
print("Training step defined:", step_train.name)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedoc

XGBoost image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:3.0-5
Training step defined: TrainXGBoostModel


## 7. Evaluation Script (Quality Gates)

This script runs in the pipeline's Evaluation step and reproduces the team's automated evaluation (`automated-model-evaluation.ipynb`):

1. **Threshold tuning:** on the validation set, picks the decision threshold with the highest precision that still reaches recall ≥ 0.70 (the same method as `02-initialize-model.ipynb`)
2. **Test metrics:** accuracy, PR-AUC, ROC-AUC, recall, precision, F1, and the confusion matrix at that threshold
3. **Benchmark:** a majority-class baseline that always predicts "not readmitted"
4. **Quality gates:** the team's 6 gates, plus a **RecallGap** fairness gate. Recall is computed per race group and per gender group, and the largest gap must stay at or below `MaxRecallGap`. Groups with fewer than 100 positive cases are excluded because their recall is too noisy to compare.

Results go to `evaluation.json`, which includes a single `gates_passed` flag (1 = all gates passed) that the Condition step reads.

> Note: `%%writefile` scripts run in their own container, so each script has its own imports.


In [20]:
%%writefile evaluate.py
import argparse
import json
import os
import tarfile

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score,
)

BASE = "/opt/ml/processing"
TARGET_RECALL = 0.70
MIN_GROUP_POSITIVES = 100


def load_split(name):
    data = np.loadtxt(f"{BASE}/{name}/{name}.csv", delimiter=",")
    return data[:, 0].astype(int), xgb.DMatrix(data[:, 1:])


def metrics(y, prob, pred):
    return {
        "accuracy": float(accuracy_score(y, pred)),
        "pr_auc": float(average_precision_score(y, prob)),
        "roc_auc": float(roc_auc_score(y, prob)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
    }


def recall_gap(df, column):
    rows = {}
    for group, g in df.groupby(column):
        positives = int(g["y_true"].sum())
        if positives >= MIN_GROUP_POSITIVES:
            rows[str(group)] = float(recall_score(g["y_true"], g["y_pred"], zero_division=0))
    gap = max(rows.values()) - min(rows.values()) if len(rows) > 1 else 0.0
    return gap, rows


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--min-recall", type=float, default=0.70)
    parser.add_argument("--min-pr-auc", type=float, default=0.20)
    parser.add_argument("--min-roc-auc", type=float, default=0.65)
    parser.add_argument("--min-precision", type=float, default=0.15)
    parser.add_argument("--max-recall-gap", type=float, default=0.10)
    args = parser.parse_args()

    with tarfile.open(f"{BASE}/model/model.tar.gz") as tar:
        tar.extractall(path=".")
    booster = xgb.Booster()
    booster.load_model("xgboost-model")

    # 1. Threshold tuning on validation (same as train-model.ipynb)
    y_val, d_val = load_split("validation")
    val_prob = booster.predict(d_val)
    prec, rec, thr = precision_recall_curve(y_val, val_prob)
    table = pd.DataFrame({"threshold": thr, "precision": prec[:-1], "recall": rec[:-1]})
    eligible = table[table["recall"] >= TARGET_RECALL]
    if len(eligible):
        threshold = float(eligible.sort_values(["precision", "threshold"], ascending=False).iloc[0]["threshold"])
    else:
        table["f1"] = 2 * table["precision"] * table["recall"] / (table["precision"] + table["recall"] + 1e-12)
        threshold = float(table.sort_values("f1", ascending=False).iloc[0]["threshold"])

    # 2. Test metrics
    y_test, d_test = load_split("test")
    test_prob = booster.predict(d_test)
    test_pred = (test_prob >= threshold).astype(int)
    test_m = metrics(y_test, test_prob, test_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, test_pred, labels=[0, 1]).ravel()

    # 3. Majority-class benchmark
    bench_pred = np.zeros(len(y_test), dtype=int)
    bench_m = metrics(y_test, bench_pred.astype(float), bench_pred)

    # 4. RecallGap fairness check
    sensitive = pd.read_csv(f"{BASE}/sensitive/test_sensitive.csv", keep_default_na=False)
    sensitive["y_true"], sensitive["y_pred"] = y_test, test_pred
    race_gap, race_recall = recall_gap(sensitive, "race")
    gender_gap, gender_recall = recall_gap(sensitive, "gender")
    worst_gap = max(race_gap, gender_gap)

    gates = [
        {"check": "Recall", "actual": test_m["recall"], "rule": f">= {args.min_recall}", "passed": test_m["recall"] >= args.min_recall},
        {"check": "PR-AUC", "actual": test_m["pr_auc"], "rule": f">= {args.min_pr_auc}", "passed": test_m["pr_auc"] >= args.min_pr_auc},
        {"check": "ROC-AUC", "actual": test_m["roc_auc"], "rule": f">= {args.min_roc_auc}", "passed": test_m["roc_auc"] >= args.min_roc_auc},
        {"check": "Precision", "actual": test_m["precision"], "rule": f">= {args.min_precision}", "passed": test_m["precision"] >= args.min_precision},
        {"check": "Beat benchmark PR-AUC", "actual": test_m["pr_auc"], "rule": f"> {bench_m['pr_auc']:.4f}", "passed": test_m["pr_auc"] > bench_m["pr_auc"]},
        {"check": "Beat benchmark recall", "actual": test_m["recall"], "rule": f"> {bench_m['recall']:.4f}", "passed": test_m["recall"] > bench_m["recall"]},
        {"check": "RecallGap (race, gender)", "actual": worst_gap, "rule": f"<= {args.max_recall_gap}", "passed": worst_gap <= args.max_recall_gap},
    ]
    gates_passed = int(all(g["passed"] for g in gates))

    report = {
        "binary_classification_metrics": {
            **{k: {"value": v, "standard_deviation": "NaN"} for k, v in test_m.items()},
            "confusion_matrix": {"0": {"0": int(tn), "1": int(fp)}, "1": {"0": int(fn), "1": int(tp)}},
        },
        "gates_passed": gates_passed,
        "decision_threshold": threshold,
        "quality_gates": [{**g, "passed": bool(g["passed"])} for g in gates],
        "benchmark_metrics": bench_m,
        "fairness": {
            "race_recall": race_recall, "race_gap": race_gap,
            "gender_recall": gender_recall, "gender_gap": gender_gap,
        },
    }

    os.makedirs(f"{BASE}/evaluation", exist_ok=True)
    with open(f"{BASE}/evaluation/evaluation.json", "w") as f:
        json.dump(report, f, indent=2)
    print(json.dumps(report, indent=2))

Overwriting evaluate.py


## 8. Evaluation Step

Runs `evaluate.py` inside the same XGBoost container used for training (3.0-5), so library versions match. Its inputs:

- The **model artifact** from the Training step
- The **validation split**, for threshold tuning
- The **test split**, for final metrics
- The **test sensitive attributes**, for the RecallGap fairness gate

Gate thresholds come from the pipeline parameters, so they can be changed at run time. The output, `evaluation.json`, is registered as a **PropertyFile** so the Condition step can read `gates_passed`.


In [ ]:
eval_processor = ScriptProcessor(
    image_uri=xgb_image,
    command=["python3"],
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="diabetes-xgb-eval",
    sagemaker_session=pipeline_session,
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

proc_outputs = step_process.properties.ProcessingOutputConfig.Outputs

eval_args = eval_processor.run(
    code="evaluate.py",
    arguments=[
        "--min-recall",
        min_recall.to_string(),
        "--min-pr-auc",
        min_pr_auc.to_string(),
        "--min-roc-auc",
        min_roc_auc.to_string(),
        "--min-precision",
        min_precision.to_string(),
        "--max-recall-gap",
        max_recall_gap.to_string(),
    ],
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=proc_outputs["validation"].S3Output.S3Uri,
            destination="/opt/ml/processing/validation",
        ),
        ProcessingInput(
            source=proc_outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
        ProcessingInput(
            source=proc_outputs["sensitive"].S3Output.S3Uri,
            destination="/opt/ml/processing/sensitive",
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation", source="/opt/ml/processing/evaluation"
        )
    ],
)

step_eval = ProcessingStep(
    name="EvaluateModel",
    step_args=eval_args,
    property_files=[evaluation_report],
    cache_config=cache_config,
)

print("Evaluation step defined:", step_eval.name)

/opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:138: SageMakerV2DeprecationWarning: ScriptProcessor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `DataProcessor` (`from sagemaker.mlops.processing import DataProcessor`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Evaluation step defined: EvaluateModel


## 9. Register Model Step

Registers the trained model as a new version in the SageMaker **Model Registry** under the `CarePath-Diabetes-Readmission` group. Each version stores:

- The model artifact and XGBoost container image
- The evaluation metrics from `evaluation.json`, viewable in the Registry UI
- An approval status (`PendingManualApproval` by default), so a human reviews the model before it's deployed

This step runs only if the model passes all seven quality gates in the next step.


In [ ]:
model_package_group_name = "CarePath-Diabetes-Readmission"

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/",
            values=[
                step_eval.properties.ProcessingOutputConfig.Outputs[
                    "evaluation"
                ].S3Output.S3Uri,
                "evaluation.json",
            ],
        ),
        content_type="application/json",
    )
)

model = Model(
    image_uri=xgb_image,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session,
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
    description="XGBoost 30-day readmission model trained by CI/CD pipeline",
)

step_register = ModelStep(name="RegisterDiabetesModel", step_args=register_args)

print("Register step defined")

/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Register step defined


## 10. Condition Step (Quality Gate)

Reads `gates_passed` from `evaluation.json`. It equals 1 only if **all 7 gates** pass: recall, PR-AUC, ROC-AUC, precision, beating both benchmark checks, and RecallGap.

- **All gates pass:** the model is registered in the Model Registry as PendingManualApproval
- **Any gate fails:** the pipeline stops with a **Fail** step, and nothing is registered. The detailed gate results stay in `evaluation.json` for review.

This gate is the core of our CI/CD process: only models that meet both the performance and the fairness bar move toward deployment.


In [23]:
step_fail = FailStep(
    name="QualityGatesFailed",
    error_message="One or more quality gates failed (performance or RecallGap). Model not registered. See evaluation.json for details.",
)

cond_gates = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="gates_passed",
    ),
    right=1,
)

step_cond = ConditionStep(
    name="CheckQualityGates",
    conditions=[cond_gates],
    if_steps=[step_register],
    else_steps=[step_fail],
)

print("Condition step defined:", step_cond.name)

Condition step defined: CheckQualityGates


## 11. Assemble and Register the Pipeline

We combine all the steps into one SageMaker Pipeline. SageMaker builds the DAG automatically from the data dependencies between steps.

**DAG:** ExtractFromFeatureStore → PreprocessDiabetesData → TrainXGBoostModel → EvaluateModel → CheckQualityGates → (RegisterDiabetesModel | QualityGatesFailed)

`upsert()` creates the pipeline the first time and updates it on later runs. This is what lets CI/CD push new versions of the pipeline definition.


In [24]:
pipeline_name = "DiabetesReadmissionPipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        feature_group_table,
        processing_instance_type,
        training_instance_type,
        min_recall,
        min_pr_auc,
        min_roc_auc,
        min_precision,
        max_recall_gap,
        model_approval_status,
    ],
    steps=[step_extract, step_process, step_train, step_eval, step_cond],
    sagemaker_session=pipeline_session,
)

definition = json.loads(pipeline.definition())
for s in definition["Steps"]:
    print(f"{s['Name']:<25} {s['Type']}")

response = pipeline.upsert(role_arn=role)
print("\nPipeline ARN:", response["PipelineArn"])

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline.py:119: SageMakerV2DeprecationWarning: Pipeline is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `Pipeline` (`from sagemaker.mlops.pipeline import Pipeline`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-

ExtractFromFeatureStore   Processing
PreprocessDiabetesData    Processing
TrainXGBoostModel         Training
EvaluateModel             Processing
CheckQualityGates         Condition



Pipeline ARN: arn:aws:sagemaker:us-east-1:769592409248:pipeline/DiabetesReadmissionPipeline


In [25]:
execution = pipeline.start()
print("NEW Execution ARN:", execution.arn)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


NEW Execution ARN: arn:aws:sagemaker:us-east-1:769592409248:pipeline/DiabetesReadmissionPipeline/execution/2knnwc9fkzrq


## 12. Execute the Pipeline

We start a pipeline run using the default parameters. SageMaker runs each step in DAG order and starts managed jobs for Processing, Training, and Evaluation. A full run takes about 15 to 20 minutes.


In [ ]:
steps = execution.list_steps()
for s in steps:
    print(f"{s['StepName']:<28} {s['StepStatus']}")

failed = [s for s in steps if s["StepStatus"] == "Failed"]
if failed:
    f = failed[0]
    print("\nFAILED STEP:", f["StepName"])
    print("Reason:", f.get("FailureReason", "none")[:1000])

    meta = f.get("Metadata", {})
    if "ProcessingJob" in meta:
        job, group = (
            meta["ProcessingJob"]["Arn"].split("/")[-1],
            "/aws/sagemaker/ProcessingJobs",
        )
    elif "TrainingJob" in meta:
        job, group = (
            meta["TrainingJob"]["Arn"].split("/")[-1],
            "/aws/sagemaker/TrainingJobs",
        )
    else:
        job = None

    if job:
        logs = boto3.client("logs")
        for st in logs.describe_log_streams(
            logGroupName=group, logStreamNamePrefix=job
        )["logStreams"]:
            for e in logs.get_log_events(
                logGroupName=group,
                logStreamName=st["logStreamName"],
                startFromHead=False,
                limit=25,
            )["events"]:
                print(e["message"])

ExtractFromFeatureStore      Executing


In [27]:
objs = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket):
    for o in page.get("Contents", []):
        if o["Key"].endswith("evaluation.json"):
            objs.append(o)

latest = max(objs, key=lambda o: o["LastModified"])
print("File:", latest["Key"], "|", latest["LastModified"])

body = s3.get_object(Bucket=bucket, Key=latest["Key"])["Body"].read()
print(json.dumps(json.loads(body), indent=2))

File: carepath-analytics/model-artifacts/evaluation/evaluation.json | 2026-10-16 23:22:04+00:00
{
  "model": "XGBoost",
  "primary_metrics": [
    "recall",
    "precision_recall_auc"
  ],
  "test_metrics": {
    "pr_auc": 0.2276555638786331,
    "roc_auc": 0.684938090089347,
    "recall": 0.7185273159144893,
    "precision": 0.1626562710041672,
    "f1": 0.2652636194234353,
    "threshold": 0.44392746686935425
  },
  "decision_threshold": 0.44392746686935425,
  "selected_candidate": 2,
  "raw_model_features": [
    "age",
    "admission_type",
    "discharge_disposition",
    "admission_source",
    "diag_1_group",
    "diag_2_group",
    "diag_3_group",
    "max_glu_serum",
    "A1Cresult",
    "change",
    "diabetesMed",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
  ],
  "transformed_feature_count": 104
}


## 13. Verify Pipeline Results

We check the latest execution's step statuses, the results of all 7 quality gates, the fairness breakdown, and the Model Registry. Each passing run adds a new version to `CarePath-Diabetes-Readmission` with status PendingManualApproval.


In [28]:
print("Working dir:", os.getcwd())
!grep -n "Athena lowercases" preprocess.py

Working dir: /home/sagemaker-user/diabetes-readmission-prediction/notebooks/setup
29:    # Athena lowercases column names; restore the original names


In [ ]:
latest = sm.list_pipeline_executions(
    PipelineName=pipeline_name, SortBy="CreationTime", SortOrder="Descending"
)["PipelineExecutionSummaries"][0]
exec_arn = latest["PipelineExecutionArn"]
print(
    "Execution:", exec_arn.split("/")[-1], "|", latest["PipelineExecutionStatus"], "\n"
)

steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=exec_arn)[
    "PipelineExecutionSteps"
]
for s in reversed(steps):
    print(f"{s['StepName']:<28} {s['StepStatus']}")

# Quality gate results from evaluation.json
eval_step = [s for s in steps if s["StepName"] == "EvaluateModel"][0]
job = sm.describe_processing_job(
    ProcessingJobName=eval_step["Metadata"]["ProcessingJob"]["Arn"].split("/")[-1]
)
uri = (
    job["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
    + "/evaluation.json"
)
b, k = uri.replace("s3://", "").split("/", 1)
report = json.loads(s3.get_object(Bucket=b, Key=k)["Body"].read())

print(f"\nDecision threshold: {report['decision_threshold']:.4f}")
print("\nQuality gates:")
print(pd.DataFrame(report["quality_gates"]).round(4).to_string(index=False))
print("\nOverall:", "PASS" if report["gates_passed"] else "FAIL")

print("\nFairness (recall by group):")
print(
    "  Race:  ",
    {k: round(v, 4) for k, v in report["fairness"]["race_recall"].items()},
    "| gap",
    round(report["fairness"]["race_gap"], 4),
)
print(
    "  Gender:",
    {k: round(v, 4) for k, v in report["fairness"]["gender_recall"].items()},
    "| gap",
    round(report["fairness"]["gender_gap"], 4),
)

# Model Registry
print("\nModel Registry:")
for p in sm.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime",
    SortOrder="Descending",
)["ModelPackageSummaryList"]:
    print(
        f"  Version {p['ModelPackageVersion']:<3} {p['ModelApprovalStatus']:<25} {p['CreationTime']:%Y-%m-%d %H:%M}"
    )

Execution: n2qwdglw0htd | Succeeded 

ExtractFromFeatureStore      Succeeded
PreprocessDiabetesData       Succeeded
TrainXGBoostModel            Succeeded
EvaluateModel                Succeeded
CheckQualityGates            Succeeded
RegisterDiabetesModel-RegisterModel Succeeded

Decision threshold: 0.4360

Quality gates:
                   check  actual     rule  passed
                  Recall  0.7197   >= 0.7    True
                  PR-AUC  0.2240   >= 0.2    True
                 ROC-AUC  0.6842  >= 0.65    True
               Precision  0.1605  >= 0.15    True
   Beat benchmark PR-AUC  0.2240 > 0.1102    True
   Beat benchmark recall  0.7197 > 0.0000    True
RecallGap (race, gender)  0.0472   <= 0.1    True

Overall: PASS

Fairness (recall by group):
  Race:   {'AfricanAmerican': 0.6981, 'Caucasian': 0.7244} | gap 0.0263
  Gender: {'Female': 0.7405, 'Male': 0.6932} | gap 0.0472

Model Registry:
  Version 4   PendingManualApproval     2026-10-16 04:44
  Version 3   PendingManualAp

## 14. Results

The pipeline trains from the Feature Store, passes all 7 quality gates, and registers each passing model as a new version in `CarePath-Diabetes-Readmission` (PendingManualApproval).

| Metric             | Team model (v1, manual notebook) | Pipeline (v4, automated) |
| ------------------ | -------------------------------- | ------------------------ |
| ROC-AUC            | 0.684                            | 0.684                    |
| PR-AUC             | 0.229                            | 0.224                    |
| Recall             | 0.725                            | 0.720                    |
| Precision          | 0.161                            | 0.161                    |
| Decision threshold | 0.435                            | 0.436                    |

**Quality gates (all passed)**
| Gate | Actual | Rule |
|---|---|---|
| Recall | 0.720 | ≥ 0.70 |
| PR-AUC | 0.224 | ≥ 0.20 |
| ROC-AUC | 0.684 | ≥ 0.65 |
| Precision | 0.161 | ≥ 0.15 |
| Beat benchmark PR-AUC | 0.224 | > 0.110 |
| Beat benchmark recall | 0.720 | > 0.000 |
| RecallGap (race, gender) | 0.047 | ≤ 0.10 |

**Fairness (recall by group)**

- Race: Caucasian 0.724, African American 0.698 (gap 0.026)
- Gender: Female 0.741, Male 0.693 (gap 0.047)
- Smaller groups (fewer than 100 positive cases) are excluded from the gap check because their recall is too noisy to compare. Age groups vary more and should be monitored separately.

**Notes**

- The pipeline reproduces the manually trained team model within about 0.005 on every metric. The small differences come from the SageMaker built-in XGBoost container versus the local XGBoost library.
- Moving training into the pipeline makes it repeatable and auditable: every run pulls the latest Feature Store data, applies the same gates, and creates a new model version for human review.
- Debugging along the way: a missing import, Athena's lowercase column names, and an indentation error each failed the Preprocess step. Every failure was caught by the DAG before training ran and diagnosed through CloudWatch logs.
